# NOVA 03 — MOD-05 Face Embedding (MobileFaceNet)
**Attach datasets:**
- `yakhyokhuja/vggface2-112x112` — VGGFace2 pre-aligned to 112x112 (exactly MobileFaceNet's input size; no alignment preprocessing needed)
- `jessicali9530/lfw-dataset` (evaluation only)

**Accelerator:** GPU. Training on a subset of identities ~6-10h.
Teacher (InsightFace ArcFace) downloads its weights on first run — internet must be ON in notebook settings.

In [ ]:
# ── NOVA bootstrap: clone repo + resolve HF token from Kaggle secret ──
# Before running: Add-ons > Secrets > attach a secret named HF_TOKEN
# (your HuggingFace write token). Settings > Accelerator > GPU T4/P100.
import os, sys, subprocess

REPO = 'https://github.com/BertinAm/nova-ml.git'
if not os.path.exists('/kaggle/working/nova-ml'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/nova-ml'], check=True)
else:  # already cloned in this session — pull latest fixes
    subprocess.run(['git', '-C', '/kaggle/working/nova-ml', 'pull'], check=True)
os.chdir('/kaggle/working/nova-ml')
sys.path.insert(0, '/kaggle/working/nova-ml/scripts')

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['NOVA_HF_USERNAME'] = 'unixio'

# GPU compatibility guard: Kaggle's PyTorch 2.10 image dropped sm_60 (P100).
# If you get a P100, switch Settings > Accelerator to 'GPU T4 x2'.
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    assert cap >= (7, 0), (
        f'{name} (sm_{cap[0]}{cap[1]}) is NOT supported by this PyTorch build. '
        'Switch Settings > Accelerator to GPU T4 x2 and restart.')
    print(f'GPU OK: {name}')
else:
    raise RuntimeError('No GPU — set Settings > Accelerator to GPU T4 x2.')
print('Bootstrap OK — repo cloned, HF token loaded.')

In [ ]:
!pip install -q insightface onnxruntime-gpu onnx2tf onnx

In [ ]:
# Auto-resolve mounts (Kaggle nests datasets 3 levels deep) and locate the
# identity-folder root: the directory whose children are identity folders.
import glob, os
inputs = (glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*')
          + glob.glob('/kaggle/input/*/*/*'))
VGG_ROOT = next(p for p in inputs if 'vggface' in p.split('/')[-1].lower())
print('VGG_ROOT =', VGG_ROOT)
!find {VGG_ROOT} -maxdepth 2 -type d | head -10
# Pick the deepest dir that contains many subfolders (identities)
VGG_DATA = None
for cand in [VGG_ROOT] + glob.glob(f'{VGG_ROOT}/*') + glob.glob(f'{VGG_ROOT}/*/*'):
    if os.path.isdir(cand):
        subs = [s for s in os.listdir(cand)[:50]
                if os.path.isdir(os.path.join(cand, s))]
        if len(subs) >= 40:  # many identity folders
            VGG_DATA = cand
            break
assert VGG_DATA, 'Could not locate identity-folder root — inspect the layout above'
print('VGG_DATA =', VGG_DATA, '| identities (sample count):',
      len(os.listdir(VGG_DATA)))

In [ ]:
# Reuse an already-trained checkpoint from HF if present (skips the ~6-10h
# training). Delete pytorch/face_embedding_best.pth on HF to force retrain.
import shutil
from huggingface_hub import hf_hub_download
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
try:
    p = hf_hub_download('unixio/nova-face-embedding',
                        'pytorch/face_embedding_best.pth',
                        token=os.environ['HF_TOKEN'])
    shutil.copy(p, '/kaggle/working/checkpoints/face_embedding_best.pth')
    SKIP_TRAINING = True
    print('Reusing trained checkpoint from HF — skipping training.')
except Exception as e:
    SKIP_TRAINING = False
    print('No checkpoint on HF — will train.')

In [ ]:
# MobileFaceNet + ArcFace loss + embedding-KD from InsightFace teacher.
# --max-identities 2000 keeps one run inside Kaggle's 12h session limit.
# If insightface teacher fails to load, rerun with --no-teacher.
if not SKIP_TRAINING:
    !python scripts/train_face_embedding.py --data-dir {VGG_DATA} \
        --epochs 50 --batch-size 128 --max-identities 2000
else:
    print('Skipped — using HF checkpoint.')

In [ ]:
# Build a SMALL calibration dir (200 images). Never point the converter at
# the full dataset — rglob over 3M files takes forever.
import random
CALIB = '/kaggle/working/calib'
shutil.rmtree(CALIB, ignore_errors=True)
os.makedirs(CALIB)
rng = random.Random(42)
idents = rng.sample(sorted(os.listdir(VGG_DATA)), 200)
for i, ident in enumerate(idents):
    d = os.path.join(VGG_DATA, ident)
    imgs = [f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.png'))]
    if imgs:
        shutil.copy(os.path.join(d, imgs[0]), f'{CALIB}/{i:03d}_{imgs[0]}')
print(len(os.listdir(CALIB)), 'calibration images')

In [ ]:
# Convert: tries INT8 first, falls back to float32 (never ship onnx2tf's
# fp16 — true fp16 tensors fail to load on standard TFLite runtimes).
!python scripts/convert_to_tflite.py \
    --checkpoint /kaggle/working/checkpoints/face_embedding_best.pth \
    --arch mobilefacenet --input-size 112 \
    --out /kaggle/working/exports/face_embedding_v1.tflite \
    --calib-dir {CALIB} --benchmark

In [ ]:
# Publish to HuggingFace: unixio/nova-face-embedding.
# NOTE: LFW verification eval is still TODO (needs pair-list parsing for the
# attached LFW dataset) — publish records training config only for now.
!python scripts/push_to_huggingface.py --module MOD-05-embed \
    --pytorch /kaggle/working/checkpoints/face_embedding_best.pth \
    --tflite /kaggle/working/exports/face_embedding_v1.tflite \
    --version 1.0.0